# Statistics reported in the results part of the paper


In [93]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pingouin as pg

from suppression_oc.constants import (
TIMEPOINT_ORDER, TIMEPOINT_LABELS
)

from suppression_oc.util import ci95

plt.style.use("suppression_oc.notebook")

import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [94]:
d_primes = pd.read_csv(
    "../data/signal_detection.csv"
)
d_primes_normal = d_primes[d_primes.condition == "normal"]

detection = pd.read_csv(
    "../data/tatai_et_al_detection.csv"
)

## Normal reaches ANOVA


In [95]:
anova_results = pg.rm_anova(
    data=d_primes_normal,
    dv="d_prime",
    within="timepoint_bin",
    subject="subject",
)
pg.print_table(anova_results)


ANOVA SUMMARY

Source           ddof1    ddof2       F    p-unc    p-GG-corr    ng2    eps  sphericity      W-spher    p-spher
-------------  -------  -------  ------  -------  -----------  -----  -----  ------------  ---------  ---------
timepoint_bin        7      210  34.255    0.000        0.000  0.328  0.413  False             0.011      0.000



#### Sphericity and Greenhouse-Geisser correction

In [96]:
from scipy.stats import f as f_dist


def gg_corrected_table(aov):
    """Greenhouse-Geisser corrected degrees of freedom and p values per effect."""
    return pd.DataFrame(
        [
            {
                "Source": effect["Source"],
                "F": effect["F"],
                "ddof1 (GG)": effect["ddof1"] * effect["eps"],
                "ddof2 (GG)": effect["ddof2"] * effect["eps"],
                "eps": effect["eps"],
                "p-unc": effect["p-unc"],
                "p-GG-corr": f_dist.sf(
                    effect["F"],
                    effect["ddof1"] * effect["eps"],
                    effect["ddof2"] * effect["eps"],
                ),
                "ng2": effect["ng2"],
            }
            for _, effect in aov.iterrows()
        ]
    )


print(
    anova_results[["Source", "W-spher", "p-spher", "sphericity"]].to_string(index=False)
)

gg_corrected_table(anova_results).round(3)

       Source  W-spher      p-spher  sphericity
timepoint_bin 0.011194 4.283304e-14       False


,Source,F,ddof1 (GG),ddof2 (GG),eps,p-unc,p-GG-corr,ng2
0,timepoint_bin,34.255,2.894,86.815,0.413,0.0,0.0,0.328


In [97]:
posthoc = pg.pairwise_tests(
    data=d_primes_normal,
    dv="d_prime",
    within="timepoint_bin",
    subject="subject",
    padjust="bonf",
    effsize="cohen",
)
pg.print_table(posthoc)


POST HOC TESTS

Contrast       A             B             Paired    Parametric          T     dof  alternative      p-unc    p-corr  p-adjust                BF10    cohen
-------------  ------------  ------------  --------  ------------  -------  ------  -------------  -------  --------  ----------  ----------------  -------
timepoint_bin  (-0.1, 0.0]   (-inf, -0.1]  True      True          -12.046  30.000  two-sided        0.000     0.000  bonf         15310000000.000   -1.715
timepoint_bin  (-0.1, 0.0]   (0.0, 0.2]    True      True            5.725  30.000  two-sided        0.000     0.000  bonf                6331.970    1.107
timepoint_bin  (-0.1, 0.0]   (0.2, 0.4]    True      True            3.236  30.000  two-sided        0.003     0.083  bonf                  12.702    0.636
timepoint_bin  (-0.1, 0.0]   (0.4, 0.6]    True      True            2.167  30.000  two-sided        0.038     1.000  bonf                   1.464    0.433
timepoint_bin  (-0.1, 0.0]   (0.6, 0.8]    True

## Comparison normal and initial uncertainty


In [98]:
kinematic_variables = {
    "reaction_time": "Mean reaction time [s]",
    "reach_time": "Mean movement time [s]",
    "max_v": "Mean max. velocity [m/s]",
    "t_max_v": "Mean time max. velocity [s]",
    "max_a": "Mean max. acceleration [m/s\u00b2]",
    "t_max_a": "Mean time max. acceleration [s]",
    "max_dec_a": "Mean max. deceleration [m/s\u00b2]",
    "t_max_dec_a": "Mean time max. deceleration [s]",
}

subject_means = detection.groupby(["condition", "subject"])[
    list(kinematic_variables)
].mean()


def mean_with_ci(values):
    """Mean across participants with its 95% confidence interval, as LaTeX."""
    half_width = ci95(len(values)) * values.sem()
    return (
        f"${values.mean():.3f}$ "
        f"$[{values.mean() - half_width:.3f}, {values.mean() + half_width:.3f}]$"
    )


summary_stats_formatted = pd.DataFrame(
    {
        condition: {
            label: mean_with_ci(subject_means.loc[condition, column])
            for column, label in kinematic_variables.items()
        }
        for condition in ["normal", "uncertain"]
    }
).rename(columns={"normal": r"\textit{certain}", "uncertain": r"\textit{uncertain}"})
summary_stats_formatted.index.name = "Kinematic variable"
summary_stats_formatted.columns.name = None

summary_stats_formatted

,\textit{certain},\textit{uncertain}
Kinematic variable,,
Mean reaction time [s],"$0.287$ $[0.274, 0.300]$","$0.284$ $[0.272, 0.296]$"
Mean movement time [s],"$0.671$ $[0.628, 0.714]$","$0.691$ $[0.648, 0.735]$"
Mean max. velocity [m/s],"$1.538$ $[1.421, 1.655]$","$1.490$ $[1.395, 1.585]$"
Mean time max. velocity [s],"$0.287$ $[0.263, 0.310]$","$0.298$ $[0.274, 0.322]$"
Mean max. acceleration [m/s²],"$19.902$ $[16.856, 22.949]$","$18.962$ $[16.026, 21.898]$"
Mean time max. acceleration [s],"$0.099$ $[0.083, 0.114]$","$0.105$ $[0.087, 0.122]$"
Mean max. deceleration [m/s²],"$9.201$ $[7.535, 10.868]$","$8.827$ $[7.276, 10.378]$"
Mean time max. deceleration [s],"$0.397$ $[0.369, 0.425]$","$0.414$ $[0.381, 0.446]$"


In [99]:
print(
    summary_stats_formatted.to_latex(
        escape=False,
        column_format="lll",
        caption=(
            r"Kinematic summary statistics for the \textit{certain} and "
            r"\textit{uncertain} conditions. Values are the mean across participants "
            r"($N=31$) of each participant's mean, with its 95\% confidence interval."
        ),
        label="tab:kinematics_summary",
    )
)

\begin{table}
\caption{Kinematic summary statistics for the \textit{certain} and \textit{uncertain} conditions. Values are the mean across participants ($N=31$) of each participant's mean, with its 95\% confidence interval.}
\label{tab:kinematics_summary}
\begin{tabular}{lll}
\toprule
 & \textit{certain} & \textit{uncertain} \\
Kinematic variable &  &  \\
\midrule
Mean reaction time [s] & $0.287$ $[0.274, 0.300]$ & $0.284$ $[0.272, 0.296]$ \\
Mean movement time [s] & $0.671$ $[0.628, 0.714]$ & $0.691$ $[0.648, 0.735]$ \\
Mean max. velocity [m/s] & $1.538$ $[1.421, 1.655]$ & $1.490$ $[1.395, 1.585]$ \\
Mean time max. velocity [s] & $0.287$ $[0.263, 0.310]$ & $0.298$ $[0.274, 0.322]$ \\
Mean max. acceleration [m/s²] & $19.902$ $[16.856, 22.949]$ & $18.962$ $[16.026, 21.898]$ \\
Mean time max. acceleration [s] & $0.099$ $[0.083, 0.114]$ & $0.105$ $[0.087, 0.122]$ \\
Mean max. deceleration [m/s²] & $9.201$ $[7.535, 10.868]$ & $8.827$ $[7.276, 10.378]$ \\
Mean time max. deceleration [s] & $

### Reaction time


In [100]:
mean_reaction_times = detection.groupby(["condition", "subject"])[
    "reaction_time"
].mean()
mean_reaction_times = mean_reaction_times.reset_index(name="mean_reaction_time")
pg.ttest(
    mean_reaction_times.loc[
        mean_reaction_times.condition == "normal", "mean_reaction_time"
    ],
    mean_reaction_times.loc[
        mean_reaction_times.condition == "uncertain", "mean_reaction_time"
    ],
    paired=True,
)

,T,dof,alternative,p-val,CI95%,cohen-d,BF10,power
T-test,0.646034,30,two-sided,0.523169,"[-0.01, 0.01]",0.088828,0.232,0.076664


### Movement time


In [101]:
detection["reach_time"].describe()

count    13589.000000
mean         0.681528
std          0.137719
min          0.190617
25%          0.586915
50%          0.678857
75%          0.774406
max          1.784432
Name: reach_time, dtype: float64

In [102]:
mean_movement_times = detection.groupby(["condition", "subject"])["reach_time"].mean()
mean_movement_times = mean_movement_times.reset_index(name="mean_movement_time")
pg.ttest(
    mean_movement_times.loc[
        mean_movement_times.condition == "normal", "mean_movement_time"
    ],
    mean_movement_times.loc[
        mean_movement_times.condition == "uncertain", "mean_movement_time"
    ],
    paired=True,
    alternative="two-sided",
)

,T,dof,alternative,p-val,CI95%,cohen-d,BF10,power
T-test,-2.361136,30,two-sided,0.024911,"[-0.04, -0.0]",0.172523,2.077,0.153455


In [103]:
print(
    "Mean diff:",
    np.mean(
        mean_movement_times.loc[
            mean_movement_times.condition == "normal", "mean_movement_time"
        ].values
        - mean_movement_times.loc[
            mean_movement_times.condition == "uncertain", "mean_movement_time"
        ].values
    ),
)

Mean diff: -0.02020270432391447


### Maximum velocity


In [104]:
mean_max_velocities = detection.groupby(["condition", "subject"])["max_v"].mean()
mean_max_velocities = mean_max_velocities.reset_index(name="mean_max_velocity")
pg.ttest(
    mean_max_velocities.loc[
        mean_max_velocities.condition == "normal", "mean_max_velocity"
    ],
    mean_max_velocities.loc[
        mean_max_velocities.condition == "uncertain", "mean_max_velocity"
    ],
    paired=True,
)

,T,dof,alternative,p-val,CI95%,cohen-d,BF10,power
T-test,1.381149,30,two-sided,0.177435,"[-0.02, 0.12]",0.166038,0.453,0.145623


### Time point of maximum velocity


In [105]:
mean_max_v_times = detection.groupby(["condition", "subject"])["t_max_v"].mean()
mean_max_v_times = mean_max_v_times.reset_index(name="mean_t_max_velocity")
pg.ttest(
    mean_max_v_times.loc[mean_max_v_times.condition == "normal", "mean_t_max_velocity"],
    mean_max_v_times.loc[
        mean_max_v_times.condition == "uncertain", "mean_t_max_velocity"
    ],
    paired=True,
)

,T,dof,alternative,p-val,CI95%,cohen-d,BF10,power
T-test,-2.037884,30,two-sided,0.050464,"[-0.02, 0.0]",0.17102,1.173,0.151611


### Endpoint error


In [106]:
mean_vertical_errors = detection.groupby(["condition", "subject"])[
    "vertical_error"
].mean()
mean_vertical_errors = mean_vertical_errors.reset_index(name="mean_vertical_error")
pg.ttest(
    mean_vertical_errors.loc[
        mean_vertical_errors.condition == "normal", "mean_vertical_error"
    ],
    mean_vertical_errors.loc[
        mean_vertical_errors.condition == "uncertain", "mean_vertical_error"
    ],
    paired=True,
)

,T,dof,alternative,p-val,CI95%,cohen-d,BF10,power
T-test,-0.976049,30,two-sided,0.336845,"[-0.0, 0.0]",0.095892,0.296,0.081142


### Detection ANOVA


In [107]:
anova_results = pg.rm_anova(
    data=d_primes,
    dv="d_prime",
    within=["timepoint_bin", "condition"],
    subject="subject",
)
pg.print_table(anova_results)


ANOVA SUMMARY

Source                          SS    ddof1    ddof2      MS       F    p-unc    p-GG-corr    ng2    eps
-------------------------  -------  -------  -------  ------  ------  -------  -----------  -----  -----
timepoint_bin              270.243        7      210  38.606  40.162    0.000        0.000  0.333  0.364
condition                   13.881        1       30  13.881   8.347    0.007        0.007  0.025  1.000
timepoint_bin * condition    0.670        7      210   0.096   0.553    0.794        0.732  0.001  0.698



#### Sphericity and Greenhouse-Geisser correction

In [108]:
d_primes_per_bin = d_primes.groupby(["subject", "timepoint_bin"], as_index=False)[
    "d_prime"
].mean()

sphericity_tests = {
    "timepoint_bin": pg.sphericity(
        d_primes_per_bin, dv="d_prime", subject="subject", within="timepoint_bin"
    ),
    "timepoint_bin * condition": pg.sphericity(
        d_primes, dv="d_prime", subject="subject", within=["condition", "timepoint_bin"]
    ),
}

for effect, result in sphericity_tests.items():
    print(
        f"{effect:26s} W = {result.W:.4f}, chi2({result.dof}) = {result.chi2:.2f}, "
        f"p = {result.pval:.3g}, sphericity = {result.spher}"
    )
print(f"{'condition':26s} only two levels, so sphericity holds by construction")

gg_corrected_table(anova_results).round(3)

timepoint_bin              W = 0.0040, chi2(27) = 151.32, p = 5.17e-19, sphericity = False
timepoint_bin * condition  W = 0.2455, chi2(27) = 38.55, p = 0.072, sphericity = True
condition                  only two levels, so sphericity holds by construction


,Source,F,ddof1 (GG),ddof2 (GG),eps,p-unc,p-GG-corr,ng2
0,timepoint_bin,40.162,2.550,76.510,0.364,0.000,0.000,0.333
1,condition,8.347,1.000,30.000,1.000,0.007,0.007,0.025
2,timepoint_bin * condition,0.553,4.889,146.682,0.698,0.794,0.732,0.001


In [109]:
posthoc = pg.pairwise_tests(
    data=d_primes,
    dv="d_prime",
    within=["timepoint_bin", "condition"],
    subject="subject",
    padjust="bonf",
    effsize="cohen"
)
# posthoc
pg.print_table(posthoc)


POST HOC TESTS

Contrast                   timepoint_bin    A             B             Paired    Parametric          T     dof  alternative      p-unc    p-corr  p-adjust                  BF10    cohen
-------------------------  ---------------  ------------  ------------  --------  ------------  -------  ------  -------------  -------  --------  ----------  ------------------  -------
timepoint_bin              -                (-0.1, 0.0]   (-inf, -0.1]  True      True          -13.777  30.000  two-sided        0.000     0.000  bonf          405900000000.000   -1.965
timepoint_bin              -                (-0.1, 0.0]   (0.0, 0.2]    True      True            6.554  30.000  two-sided        0.000     0.000  bonf                 53900.000    1.214
timepoint_bin              -                (-0.1, 0.0]   (0.2, 0.4]    True      True            3.852  30.000  two-sided        0.001     0.016  bonf                    53.819    0.827
timepoint_bin              -                (-0.

### SI Posthoc comparison table

In [110]:
cond_posthoc = (
    posthoc[posthoc["Contrast"] == "timepoint_bin * condition"]
    .set_index("timepoint_bin")
    .loc[TIMEPOINT_ORDER]
)

# pingouin contrasts certain (A) against uncertain (B). The table reports the
# uncertain - certain direction used throughout the Results text, so t and
# Cohen's d are sign-flipped to match the sign of Delta d'.
delta_paired = {
    b: (
        d_primes[d_primes.timepoint_bin == b]
        .pivot_table(index="subject", columns="condition", values="d_prime")
        .pipe(lambda p: p["uncertain"] - p["normal"])
    )
    for b in TIMEPOINT_ORDER
}
delta_d_prime = {b: delta_paired[b].mean() for b in TIMEPOINT_ORDER}
delta_d_prime_ci = {
    b: (
        delta_paired[b].mean() - ci95(len(delta_paired[b])) * delta_paired[b].sem(),
        delta_paired[b].mean() + ci95(len(delta_paired[b])) * delta_paired[b].sem(),
    )
    for b in TIMEPOINT_ORDER
}


def _fmt_p(p):
    return "$<.001$" if p < 0.001 else f"${p:.3f}$"


posthoc_table = pd.DataFrame(
    {
        r"$\Delta d'$": [f"${delta_d_prime[b]:.3f}$" for b in TIMEPOINT_ORDER],
        r"$95\% CI_{\Delta d'}$": [
            f"$[{delta_d_prime_ci[b][0]:.3f}, {delta_d_prime_ci[b][1]:.3f}]$"
            for b in TIMEPOINT_ORDER
        ],
        r"$\textit{t}(30)$": [f"${-cond_posthoc.loc[b, 'T']:.3f}$" for b in TIMEPOINT_ORDER],
        r"$\textit{d}$": [f"${-cond_posthoc.loc[b, 'cohen']:.3f}$" for b in TIMEPOINT_ORDER],
        r"$p_\text{Bonf}$": [
            _fmt_p(cond_posthoc.loc[b, "p-corr"]) for b in TIMEPOINT_ORDER
        ],
        r"$BF_{10}$": [
            f"${float(cond_posthoc.loc[b, 'BF10']):.2f}$" for b in TIMEPOINT_ORDER
        ],
    },
    index=list(map(lambda x: f"${x}$", TIMEPOINT_LABELS)),
)
posthoc_table.index.name = "Stimulation time bin"

print(
    posthoc_table.to_latex(
        escape=False,
        column_format="lllllll",
        caption=(
            r"Per-bin paired comparisons of tactile sensitivity ($d'$) between the "
            r"\textit{certain} and \textit{uncertain} conditions "
            r"($\Delta d' = d'_\text{uncertain}-d'_\text{certain}$, $N=31$), with its "
            r"95\% confidence interval, Cohen's $d$, Bonferroni-corrected "
            r"$p$ values, and Bayes factors $BF_{10}$."
        ),
        label="tab:uncertainty_posthoc",
    )
)

\begin{table}
\caption{Per-bin paired comparisons of tactile sensitivity ($d'$) between the \textit{certain} and \textit{uncertain} conditions ($\Delta d' = d'_\text{uncertain}-d'_\text{certain}$, $N=31$), with its 95\% confidence interval, Cohen's $d$, Bonferroni-corrected $p$ values, and Bayes factors $BF_{10}$.}
\label{tab:uncertainty_posthoc}
\begin{tabular}{lllllll}
\toprule
 & $\Delta d'$ & $95\% CI_{\Delta d'}$ & $\textit{t}(30)$ & $\textit{d}$ & $p_\text{Bonf}$ & $BF_{10}$ \\
Stimulation time bin &  &  &  &  &  &  \\
\midrule
$[<-.1]s$ & $0.263$ & $[-0.004, 0.530]$ & $2.015$ & $0.327$ & $0.424$ & $1.13$ \\
$(-.1, .0]s$ & $0.433$ & $[0.219, 0.647]$ & $4.136$ & $0.491$ & $0.002$ & $108.19$ \\
$(.0, .2]$ & $0.455$ & $[0.192, 0.718]$ & $3.535$ & $0.656$ & $0.011$ & $25.30$ \\
$(.2, .4]$ & $0.229$ & $[-0.035, 0.493]$ & $1.769$ & $0.263$ & $0.696$ & $0.77$ \\
$(.4, .6]$ & $0.317$ & $[-0.041, 0.674]$ & $1.810$ & $0.300$ & $0.642$ & $0.82$ \\
$(.6, .8]$ & $0.363$ & $[0.012, 0.714]$ & $